# Inspect Consolidated Snow-Covered Area Datasets

Load and visualize the two SCA source datasets for early March 2000:

1. **MOD10C1 v061** — CMG (~5 km, lat/lon) daily snow cover and clear-sky percentage from MODIS. Adds the CI-bounded SCA target as defined in TM 6-B10.
2. **UA daily 4-km SWE/Snow Depth (NSIDC-0719)** — daily snow depth on EPSG:5070 over CONUS. Reinterpreted as a **second, depth-derived SCA source**: a pixel is "snow-covered" when `snow_depth > threshold` (default 1 mm). Area-weighting via gdptools yields a per-HRU fractional snow-covered area that complements MOD10C1.

This second source breaks MOD10C1's single-source lock on the SCA target and pushes coverage **back to WY 1982** (MOD10C1 is post-2000 only). On HRUs where MOD10C1 is gated out by cloud cover (CI ≤ 70%), UA SWE still contributes a fractional snow-cover value, widening multi-source min/max bounds at otherwise-blank days.

Sources (see `catalog/variables.yml` → `snow_covered_area`):
- MOD10C1 v061 `Day_CMG_Snow_Cover` — 0-100 percent snow per cell (with flag values 107, 237, 239, 250, 253, 255 that must be masked)
- MOD10C1 v061 `Day_CMG_Clear_Index` — 0-100 percent clear-sky per cell. **This is the variable TM 6-B10 calls "confidence interval"** for SCA filtering (CI > 70%). In v006 it was named `Day_CMG_Confidence_Index`; renamed in v061. Same flag values as snow cover.
- UA SWE `snow_depth` — mm, native units; fill values already masked to NaN at consolidate time. Shown raw and as the derived binary `snow_depth > 1 mm` preview.

The consolidated MOD10C1 NetCDF also carries `Day_CMG_Cloud_Obscured` (kept for QA cross-checks) and `Snow_Spatial_QA` (a 0-4 categorical QA flag, *not* the CI — see the explanation cell below for why we don't load it here).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

from _helpers import save_figure

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT_DIR = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")


# Set True (and re-run) to populate docs/figures/consolidated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name
TARGET_TIME = "2000-03-01"
TARGET_YEAR = 2000


## Load the dataset

In [ ]:
import numpy as np
import xarray as xr

mod10c1_path = DATASTORE / "mod10c1_v061" / f"mod10c1_v061_{TARGET_YEAR}_consolidated.nc"
ua_swe_path = DATASTORE / "ua_swe" / "daily" / f"ua_swe_daily_{TARGET_YEAR}.nc"

UA_SWE_DEPTH_THRESHOLD_MM = 1.0  # provisional; same default as PR-D will use


def _mask_flags(da):
    """Mask MOD10C1 flag values (>100) to NaN. Applies to percent-encoded
    variables: Day_CMG_Snow_Cover, Day_CMG_Clear_Index, Day_CMG_Cloud_Obscured.
    """
    return da.where((da >= 0) & (da <= 100))


def _identity(da):
    """No additional mask: ua_swe consolidator already NaN'd fill values."""
    return da


def _depth_to_binary(da):
    """Snow-covered binary preview at the pixel scale.

    Per-pixel ``snow_depth > threshold`` evaluates to True (1.0) or
    False (0.0); fill-coded pixels are NaN-preserved. Area-weighting
    pixel-binary values via gdptools at the aggregate stage yields the
    fractional snow-covered area per HRU. This cell only displays the
    pixel-level binary; the per-HRU fractional value lives in PR-B's
    aggregator output (``aggregate/ua_swe.py`` ``pre_aggregate_hook``).
    """
    binary = (da > UA_SWE_DEPTH_THRESHOLD_MM).where(da.notnull()).astype("float32")
    return binary


datasets = {
    "MOD10C1 v061 (Day_CMG_Snow_Cover)": {
        "path": mod10c1_path,
        "var": "Day_CMG_Snow_Cover",
        "units": "percent (0-100, flag values masked)",
        "cmap": "Blues",
        "mask": _mask_flags,
        "coord_type": "latlon",
    },
    "MOD10C1 v061 (Day_CMG_Clear_Index)": {
        "path": mod10c1_path,
        "var": "Day_CMG_Clear_Index",
        "units": "percent (0-100, flag values masked) - TM 6-B10 CI",
        "cmap": "viridis",
        "mask": _mask_flags,
        "coord_type": "latlon",
    },
    "UA SWE (snow_depth, native mm)": {
        "path": ua_swe_path,
        "var": "snow_depth",
        "units": "mm (NaN-masked fill)",
        "cmap": "Blues",
        "mask": _identity,
        "coord_type": "xy_5070",
    },
    f"UA SWE (snow_depth > {int(UA_SWE_DEPTH_THRESHOLD_MM)} mm)": {
        "path": ua_swe_path,
        "var": "snow_depth",
        "units": f"0 / 1 binary (threshold = {UA_SWE_DEPTH_THRESHOLD_MM:g} mm)",
        "cmap": "Blues",
        "mask": _depth_to_binary,
        "coord_type": "xy_5070",
    },
}

opened = {}
missing_vars = []
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    if info["var"] not in ds.data_vars:
        missing_vars.append((label, info["var"]))
        ds.close()
        continue
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")

if missing_vars:
    print()
    print("WARN: variables missing from the consolidated NC:")
    for label, var in missing_vars:
        print(f"  - {label} ({var})")
    print("  Re-run the matching fetch (`pixi run nhf-targets fetch mod10c1` or `pixi run nhf-targets fetch ua-swe`).")

## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)


## Plot March 1, 2000 — all four panels


In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    # Force a 2x2 layout so MOD10C1 (row 0) and UA SWE (row 1) are visually grouped.
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows), squeeze=False)
    flat = axes.flatten()

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = flat[idx]
        var = info["var"]
        da = info["mask"](ds[var].sel(time=TARGET_TIME, method="nearest"))
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap=info.get("cmap", "viridis"), robust=True, cbar_kwargs={"orientation": "horizontal", "shrink": 0.8, "pad": 0.08, "aspect": 30})
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=10)
        if info.get("coord_type") == "xy_5070":
            ax.set_xlabel("x (m, EPSG:5070)")
            ax.set_ylabel("y (m, EPSG:5070)")
        else:
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
        ax.set_aspect("equal")

    for idx in range(n, nrows * ncols):
        flat[idx].set_visible(False)

    fig.suptitle(
        f"SCA consolidated sources — MOD10C1 v061 + UA SWE depth — nearest to {TARGET_TIME}",
        fontsize=14, y=1.01,
    )
    plt.tight_layout()
    save_figure(fig, "snow_covered_area_raw_panels")
    plt.show()


### Adding a depth-derived SCA source (UA SWE)

MOD10C1 alone is satellite-only, and its useful coverage at any given HRU/day is gated by the cloud-cover field — when `Day_CMG_Clear_Index ≤ 70` we drop the observation per TM 6-B10. **The UA SWE snow-depth field adds a second, independent SCA signal that is not weather-blocked:**

- Per-pixel `snow_depth > threshold_mm` (default 1 mm) evaluates to a 0/1 binary at the 4 km grid scale.
- Area-weighting the pixel-binary via gdptools (PR-B `aggregate/ua_swe.py`) yields a per-HRU fractional snow-covered area that commutes correctly with the area-weighted mean **only because the threshold is applied pre-aggregation**. Doing the threshold on the HRU-mean depth would be wrong.
- For days when MOD10C1 is gated out (CI ≤ 70), UA SWE still contributes a fractional value; the multi-source `snow_covered_area` target combines them via NaN-aware min/max (PR-D `targets/sca.py`).

**Why threshold = 1 mm.** Generous on purpose — counts thin/patchy snow that PRMS would model as snowpack-present. Configurable per project via `catalog.variables.snow_covered_area.depth_threshold_mm`. We will revisit after calibration if the multi-source bounds are systematically biased relative to MOD10C1 on overlapping days.

**Pre-2000 coverage.** UA SWE pushes the SCA target start from MOD10C1's 2000 to **WY 1982** — but pre-2000 the target collapses to a degenerate `[v, v]` interval (lower = upper, since MOD10C1 contributes nothing). The target builder will document this honestly in the output NC's `range_notes`.

### Why we don't load `Snow_Spatial_QA` here (and why an earlier version of this repo did)

TM 6-B10 (Hay et al., 2023) describes the SCA calibration target as follows:

> "The daily snow-covered area values have an associated confidence interval, where a confidence interval equal to 100 percent indicates clear sky conditions with the highest level of confidence... daily values of snow-covered area were used if the confidence interval was greater than 70 percent."

The "100 percent = clear sky" phrasing maps to MOD10C1's **clear-sky percentage** variable, not to `Snow_Spatial_QA`:

- **`Day_CMG_Clear_Index`** (v061; was `Day_CMG_Confidence_Index` in v006) is a 0-100 percent variable. The value is the percentage of the cell that was cloud-free on that day. A value of 100 means the cell was fully clear-sky - exactly the TM 6-B10 wording. **This is what we use as the CI.**
- **`Snow_Spatial_QA`** is a 0-4 categorical *quality* flag (0=best, 1=good, 2=ok, 3=poor, 4=other) describing the algorithmic confidence in the snow retrieval, not the clear-sky percentage. Despite the source HDF carrying a `units: percent` attribute, the data are not on a 0-100 scale and dividing by 100 produces meaningless "fractions".

This repo previously had `range_method: modis_ci` configured to read `ci = Snow_Spatial_QA / 100` and filter `ci > 0.70`. That filter would *only* pass values >=70 on a 0-4 scale - i.e. only the special-case flag codes (237=inland water, 239=ocean, 250=cloud-obscured, 253=not mapped, 255=fill). Every legitimate QA value (0-4) would be rejected. The catalog has been corrected to read the CI from `Day_CMG_Clear_Index` instead.

`Snow_Spatial_QA` is still consolidated into the NetCDF for any future QA cross-checks (e.g. confirming that low Clear_Index correlates with poor QA flags), but is intentionally not loaded here - there is no quantitative use for it in the SCA target as defined by TM 6-B10.

**Practical note on flag values**

`Day_CMG_Snow_Cover`, `Day_CMG_Clear_Index`, and `Day_CMG_Cloud_Obscured` all carry the same flag values for non-data cells:

- 107 = lake ice
- 111 = night
- 237 = inland water
- 239 = ocean
- 250 = cloud-obscured water
- 253 = data not mapped
- 255 = fill

Any quantitative use must mask values outside the documented `valid_range` (0-100) before averaging or thresholding. Without the mask, the domain mean of `Day_CMG_Snow_Cover` for a typical CONUS day is around 100, dominated by 237/239/250 flag codes rather than real percent-snow values.

**For the SCA target builder**

`targets/sca.py` (currently a stub) should — once promoted to implementation in PR-D:

1. Load MOD10C1 `Day_CMG_Snow_Cover` and `Day_CMG_Clear_Index`, masking flag values >100 to NaN. Compute `sca = Day_CMG_Snow_Cover / 100` and `ci = Day_CMG_Clear_Index / 100`. Discard any cell where `ci <= 0.70`.
2. Load UA SWE `snow_covered_fraction` (PR-B aggregate output: pre-aggregation `snow_depth > threshold_mm` → 0/1 → HRU-area-weighted fraction).
3. Per-HRU per-day, combine the two via NaN-aware min/max:
   - `lower = nanmin(mod10c1_lower_ci, ua_swe.snow_covered_fraction)`
   - `upper = nanmax(mod10c1_upper_ci, ua_swe.snow_covered_fraction)`
   The mod10c1 contribution is a CI-bounded interval; UA SWE contributes a degenerate single-value interval `[v, v]`.
4. Preserve TM 6-B10's July/August zero-forcing post-combine.


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
